# Integración numérica — Simpson 1/3

Función reutilizable primero, luego ejemplos de una variable, límites dependientes, integrales complejas y comprobación de convergencia. No se resuelven los enunciados de las tareas.

**Fuente:** `04-integrales, pp. 11–15` y notebook de clase original. Revisión de estudio con IA; usar en evaluación solo si el material asistido está autorizado.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Método para copiar
$h=(b-a)/N$; pesos 1, 4, 2, 4, …, 1. N debe ser par. Se integra una parábola por cada dos subintervalos; exacto hasta grado 3 en aritmética exacta.

In [2]:
def Simpson(f, a, b, N):
    if not isinstance(N, (int, np.integer)) or N < 2 or N % 2 != 0:
        raise ValueError('N debe ser un entero positivo y par')
    if a == b:
        return 0.0
    x = np.linspace(a, b, N+1)
    h = (b-a)/N
    impares = sum(f(xi) for xi in x[1:-1:2])
    pares = sum(f(xi) for xi in x[2:-1:2])
    return h/3*(f(x[0]) + 4*impares + 2*pares + f(x[-1]))

## 2. Ejemplo de clase: cambiar f, a, b, N
El ejemplo original integra 1/x. Si el límite superior cambia, llama de nuevo a la función con ese límite.

In [3]:
def f(x):
    return 1/x
a, b, N = 1, 2, 100
I = Simpson(f,a,b,N)
print(I, 'error:', abs(I-np.log(2)))
assert np.isclose(Simpson(lambda x: x**3,0,2,10),4)
assert np.isclose(Simpson(lambda x: x**3,2,0,10),-4)
assert Simpson(f,1,1,10)==0

0.6931471808723674 error: 3.1242208820003725e-10


## 3. Límites superiores variables
Para obtener una curva $F(s)=\int_a^s f(t)dt$, integra una vez por cada valor de s. La variable interna t no debe confundirse con el límite s.

In [4]:
s = np.linspace(1,2,5)
F = np.array([Simpson(f,1,si,N) for si in s])
print(np.column_stack((s,F)))
assert np.allclose(F,np.log(s))

[[1.         0.        ]
 [1.25       0.22314355]
 [1.5        0.40546511]
 [1.75       0.55961579]
 [2.         0.69314718]]


## 4. Integral iterada: ejemplo genérico
$\int_a^b dx\int_{c(x)}^{d(x)} f(x,y)dy$. Primero fija x y define g(x) como la integral interna. Después integra g. Si los límites son constantes, usa funciones constantes o números en el llamado interior. El ejemplo es un triángulo y no el enunciado de la tarea.

In [5]:
def f2(x,y):
    return x+y
a, b = 0, 1
Nx, Ny = 20, 20
def c(x):
    return 0
def d(x):
    return x
def g(x):
    return Simpson(lambda y: f2(x,y), c(x), d(x), Ny)
I2 = Simpson(g,a,b,Nx)
print(I2)
assert np.isclose(I2,0.5)

0.5


## 5. Funciones complejas y control del error
Simpson funciona también con valores complejos; aplica la parte real al resultado solo si la definición lo pide. No conviertas a float antes de sumar.

In [6]:
Icomplex = Simpson(lambda t: np.exp(1j*t),0,np.pi,100)
print(Icomplex)
assert np.allclose(Icomplex,2j,atol=1e-7)

(-1.2323802026297166e-16+2.000000010824504j)


In [7]:
# Estimación por refinamiento para una función suave
a, b, N = 1, 2, 10
I_N = Simpson(f,a,b,N)
I_2N = Simpson(f,a,b,2*N)
error_estimado = abs(I_2N-I_N)/15
print('I(2N):', I_2N, 'error estimado:', error_estimado)
print('error real de comprobación:', abs(I_2N-np.log(2)))

I(2N): 0.6931473746651161 error estimado: 1.9040158761492638e-07
error real de comprobación: 1.9410517082540935e-07


**Consejos:** la estimación /15 usa el orden global O(h⁴) en régimen asintótico; no es una cota garantizada. Comprueba otro refinamiento, especialmente si hay oscilaciones, cancelación o límites que generan falta de suavidad. Para una integral doble, refina Nx y Ny por separado. Cuatro cifras seguras requieren interpretar la tolerancia según la magnitud del resultado; no son siempre cuatro decimales. En coordenadas nuevas incluye el jacobiano.